# ATMS et résolution de conflits — les environnements et les stratégies

**Navigation** : [README](./README.md) | [7-Orchestration ->](./Argumentation-07-Orchestration-Python.ipynb)


**Objectifs d'apprentissage**

- Distinguer l'étiquette **booléenne** d'une croyance (JTMS) de l'étiquette **par environnements** (ATMS)
- Calculer l'étiquette d'une conclusion : produit cartésien des étiquettes des prémisses, filtrage par les nœuds `out`
- Dire ce que fait — et ce qu'il ne fait pas — l'invalidation d'un environnement
- Résoudre un conflit entre agents avec les cinq stratégies, et lire leurs replis (`fallback_from`) et leurs garde-fous (`ValueError`, `missing_confidence`)

**Prérequis** : le rung 6-JTMS (justifications, étiquetage, cascade de rétractation) — le lien est dans le paragraphe suivant. Aucune dépendance externe : bibliothèque standard Python uniquement, ni LLM ni JVM.

**Durée** : 40 min.

Le rung [6-JTMS](./Argumentation-06-JTMS-Python.ipynb) maintient un **booléen** par croyance : une proposition est IN, OUT ou UNDECIDED. Cela suffit quand il n'y a qu'une seule lecture du monde. Mais l'analyse argumentative multi-agents vit de plusieurs lectures concurrentes : Sherlock tient une hypothèse, Watson en tient une autre, et chacune n'est vraie que *sous certaines conditions*.

C'est le problème que résout un **ATMS** (Assumption-based Truth Maintenance System) : au lieu d'un booléen par croyance, on conserve **l'ensemble des environnements minimaux** d'hypothèses sous lesquels la croyance reste dérivable. Une conclusion n'est plus « vraie », elle est « vraie sous {p, q} et sous {p, r} mais pas sous {s} ».

Ce carnet expose deux moteurs portés dans `argumentation_lib/` :

- `_atms_core.py` — l'ATMS : environnements, justifications, invalidation, explication ;
- `_conflict_resolution.py` — le résolveur de conflits entre agents : cinq stratégies, leurs replis et leurs garde-fous.

Les deux fichiers sont des **copies verbatim** du dépôt EPITA (`argumentation_analysis/services/jtms/`, commit `713927e1`) ; le registre de provenance est `argumentation_lib/NOTICE-EPITA`. Aucune réimplémentation n'est proposée ici : le carnet mesure le moteur tel qu'il est.


In [1]:
# Imports : la serie vit dans ce dossier, `argumentation_lib` y reside a cote.
import json
import sys
from pathlib import Path

BASE = Path.cwd()
if str(BASE) not in sys.path:
    sys.path.insert(0, str(BASE))

from argumentation_lib._atms_core import ATMS, CONTRADICTION_SYMBOL
from argumentation_lib._conflict_resolution import ConflictResolver


def envs(atms, nom):
    """Etiquettes d'un noeud, triees pour une lecture stable."""
    return sorted(sorted(e) for e in atms.get_environments(nom))

## 1. Le modèle : un environnement par combinaison d'hypothèses

Une **hypothèse** est une croyance que l'on ne justifie pas mais que l'on tient pour possible. Un **environnement** est un ensemble d'hypothèses : c'est la lecture du monde sous laquelle une conclusion est derivable. L'**étiquette** d'un nœud est l'ensemble des environnements qui le dérivent.

Le moteur démarre avec un nœud spécial, `⊥` (contradiction), dont l'étiquette est vide : tant qu'aucune contradiction n'est derivée, `⊥` ne contient rien.

In [2]:
# Trois hypotheses, une conclusion derivee de deux d'entre elles.
atms = ATMS()
atms.add_assumption("p")
atms.add_assumption("q")
atms.add_node("r")
atms.add_justification(["p", "q"], [], "r")

print("p :", envs(atms, "p"))
print("q :", envs(atms, "q"))
print("r :", envs(atms, "r"))
print("hypotheses :", sorted(atms.get_assumptions()))
print("noeuds :", sorted(atms.nodes))

p : [['p']]
q : [['q']]
r : [['p', 'q']]
hypotheses : ['p', 'q']
noeuds : ['p', 'q', 'r', '⊥']


### Lecture du résultat

Chaque hypothèse porte **son propre environnement singleton** : `p` est dérivable sous `{p}`, `q` sous `{q}`. La conclusion `r` n'est pas dérivable sous `{p}` seul ni sous `{q}` seul : son étiquette est `[['p', 'q']]`, c'est-à-dire l'**union** des hypothèses de la justification. Une étiquette n'est donc pas une disjonction de lectures, c'est une conjonction d'hypothèses.

Le nœud `⊥` figure dans la liste des nœuds dès la construction : il existe avant d'être everité.

## 2. Une conclusion peut porter plusieurs environnements

L'étiquette d'une conclusion n'est pas un singleton. Le moteur prend le **produit cartésien** des étiquettes de ses prémisses, puis **fusionne** chaque combinaison en un seul environnement (l'union des hypothèses). Un nœud qui est à la fois une hypothèse et une conclusion cumule les deux étiquettes.

C'est la différence structurante avec le JTMS : là où un booléen répond « vrai ou faux », l'ATMS répond « vrai sous quelles conditions » — et ces conditions peuvent être plusieurs.

In [3]:
# "z" est a la fois derivee de "p" seule et de "p" & "s" : elle porte deux environnements.
atms2 = ATMS()
atms2.add_assumption("p")
atms2.add_assumption("s")
atms2.add_assumption("q")
atms2.add_node("z")
atms2.add_justification(["p"], [], "z")
atms2.add_justification(["p", "s"], [], "z")
atms2.add_node("v")
atms2.add_justification(["q"], [], "v")
atms2.add_node("w")
atms2.add_justification(["z", "v"], [], "w")

print("z :", envs(atms2, "z"))
print("v :", envs(atms2, "v"))
print("w :", envs(atms2, "w"))

z : [['p'], ['p', 's']]
v : [['q']]
w : [['p', 'q'], ['p', 'q', 's']]


### Lecture du résultat

`z` porte deux environnements (`['p']` et `['p', 's']`) : chacune de ses deux justifications en a déposé un. La conclusion `w` combine `z` et `v` : le produit cartésien des deux étiquettes donne deux combinaisons, dont l'union produit les deux environnements de `w`. On lit `['p', 'q']` et `['p', 'q', 's']` : l'étiquette d'une conclusion **hérite** des hypothèses de ses prémisses, et une hypothèse qui traverse une justification se propage avec elle.

Une étiquette ATMS est donc un ensemble d'environnements, pas un ensemble de booléens.

## Exercice 1 — le produit cartésien à la main

**Objectif** : calculer l'étiquette d'une conclusion dont les deux prémisses portent chacune deux environnements, et vérifier le compte.

**Consigne** : les deux prémisses `gauche` et `droite` sont construites ci-dessous ; écrivez dans `attendu` la liste des environnements de `conclusion`, puis comparez avec la mesure du moteur (la ligne `conclusion` s'affiche après votre écriture).

`# Indice` : chaque environnement de la conclusion est l'union d'un environnement de `gauche` et d'un environnement de `droite` — il y en a autant que de combinaisons.
`# Etape 1` : lister les environnements de `gauche` et de `droite` (ils s'affichent).
`# Etape 2` : écrire la liste attendue dans `attendu`.

In [4]:
# Exercice 1 : les environnements de la conclusion
atms3 = ATMS()
atms3.add_assumption("g1")
atms3.add_assumption("g2")
atms3.add_assumption("d1")
atms3.add_assumption("d2")
atms3.add_node("gauche")
atms3.add_justification(["g1"], [], "gauche")
atms3.add_justification(["g1", "g2"], [], "gauche")
atms3.add_node("droite")
atms3.add_justification(["d1"], [], "droite")
atms3.add_justification(["d1", "d2"], [], "droite")
atms3.add_node("conclusion")
atms3.add_justification(["gauche", "droite"], [], "conclusion")

attendu = None  # TODO etudiant : la liste des environnements de "conclusion"

print("gauche :", envs(atms3, "gauche"))
print("droite :", envs(atms3, "droite"))
print("Exercice a completer")

gauche : [['g1'], ['g1', 'g2']]
droite : [['d1'], ['d1', 'd2']]
Exercice a completer


## 3. L'invalidation : une perte, jamais une re-derivation

Le moteur ne sait pas **retirer** une hypothèse ; il sait **invalider un environnement**. L'invalidation retire cet environnement et tous ceux qui le contiennent, de l'étiquette de *chaque* nœud.

Le contrat est explicite et il est contre-intuitif : **les étiquettes ne grossissent qu'à l'insertion d'une justification**. Une invalidation ne redérive rien — elle ne fait que retrancher. Ce qui a été retiré reste retiré jusqu'à ce qu'une *nouvelle* justification le rederive.

In [5]:
# Une derivee simple, puis l'invalidation de l'hypothese qui la porte.
atms4 = ATMS()
atms4.add_assumption("x")
atms4.add_node("t")
atms4.add_justification(["x"], [], "t")

print("t avant :", envs(atms4, "t"))
atms4.invalidate_environment(frozenset({"x"}))
print("t apres :", envs(atms4, "t"))
print("x apres :", envs(atms4, "x"))
print("justifications de t :", len(atms4.nodes["t"].justifications))

t avant : [['x']]
t apres : []
x apres : []
justifications de t : 1


### Lecture du résultat

L'invalidation de `{x}` vide l'étiquette de `t` **et** celle de `x` : l'environnement retiré est effacé partout, y compris de l'hypothèse elle-même. En revanche la justification de `t` est toujours là (une justification). Elle ne produit aucun effet : rien ne la rejoue.

La distinction est la suivante : une invalidation est une **opération destructive sur les étiquettes**, pas un retour à un état antérieur. Le moteur ne mémorise pas d'où viennent les étiquettes — il ne sait qu'en ajouter (à l'insertion) et en retirer (à l'invalidation).

## 4. La contradiction n'est pas mémorisée

Quand une justification derive `⊥`, le moteur invalide l'environnement qui la rendait possible — et cette invalidation **efface aussi l'étiquette de `⊥` lui-même**. Un « nogood » (un ensemble d'hypothèses contradictoires) n'est donc pas durable : la contradiction est structurelle, pas mémorielle.

C'est un choix assumé, et il a une conséquence pratique : un consommateur qui veut se souvenir des contradictions rencontrées doit tenir son propre registre.

In [6]:
atms5 = ATMS()
atms5.add_assumption("m")
atms5.add_assumption("n")
atms5.add_justification(["m"], [], "n")

print("coherent {m,n} avant :", atms5.is_consistent(frozenset({"m", "n"})))
atms5.add_justification(["m", "n"], [], CONTRADICTION_SYMBOL)
print("coherent {m,n} apres :", atms5.is_consistent(frozenset({"m", "n"})))
print("⊥ :", envs(atms5, CONTRADICTION_SYMBOL))
print("n :", envs(atms5, "n"))
print("m :", envs(atms5, "m"))

coherent {m,n} avant : True
coherent {m,n} apres : True
⊥ : []
n : [['n']]
m : []


### Lecture du résultat

Avant la contradiction, `{m, n}` est cohérent. Après l'insertion de la justification qui dérive `⊥`, `{m, n}` est **encore** cohérent : l'invalidation déclenchée par `⊥` a vidé l'étiquette de `⊥` elle-même. Le moteur ne se souvient pas du conflit.

Regardez `m` : son étiquette est vide. L'invalidation a retiré `{m}` de partout — de `⊥`, de `n` (qui portait `{m}` par dérivation) et de `m`. En revanche `n` garde son propre environnement `{n}` : `{m, n}` ne contient pas `{n}`, donc `{n}` n'est pas un sur-ensemble de l'environnement invalidé et survit. L'invalidation porte sur les **sur-ensembles** de l'environnement invalidé, pas sur les hypotheses qu'il contient.

## 5. Le nœud `out` : bloquer une conclusion par sous-ensemble

Une justification peut porter une liste de nœuds `out` : la conclusion n'est derivable que si **aucun** de ces nœuds n'est dérivable sous un environnement contenu dans l'environnement candidat. Le test est un test de **sous-ensemble** : un nœud `out` dérivé sous `{u}` bloque l'environnement `{u, s}` comme il bloque `{u}`.

Un nœud `out` qui n'est pas derivable ne bloque rien — c'est le cas ordinaire.

In [7]:
# Cas 1 : le noeud "out" n'est pas derivable sous l'environnement candidat -> la conclusion tient.
atms6 = ATMS()
atms6.add_assumption("s")
atms6.add_assumption("u")
atms6.add_node("v")
atms6.add_node("w")
atms6.add_node("bloque")
atms6.add_justification(["s"], [], "v")
atms6.add_justification(["u"], [], "w")
atms6.add_justification(["v"], ["w"], "bloque")
print("cas 1 — bloque :", envs(atms6, "bloque"))

# Cas 2 : "w" est aussi derivable de "s" -> il bloque l'environnement candidat.
atms7 = ATMS()
atms7.add_assumption("s")
atms7.add_assumption("u")
atms7.add_node("v")
atms7.add_node("w")
atms7.add_node("bloque")
atms7.add_justification(["s"], [], "v")
atms7.add_justification(["s"], [], "w")
atms7.add_justification(["u"], [], "w")
atms7.add_justification(["v"], ["w"], "bloque")
print("cas 2 — bloque :", envs(atms7, "bloque"))

# Un noeud inconnu ne se devine pas : l'acces echoue bruyamment.
try:
    atms6.get_environments("inconnu")
except KeyError as e:
    print("noeud inconnu -> KeyError :", e)

cas 1 — bloque : [['s']]
cas 2 — bloque : []
noeud inconnu -> KeyError : "Node 'inconnu' not found in ATMS."


### Lecture du résultat

Au cas 1, `w` n'est dérivable que sous `{u}` : l'environnement candidat `{s}` ne contient pas `{u}`, donc rien ne bloque et `bloque` est dérivable sous `['s']`. Au cas 2, `w` est dérivable sous `{s}` : l'environnement candidat `{s}` contient `{s}`, donc la conclusion est écartée et l'étiquette de `bloque` est vide. Le filtrage se fait par **inclusion d'ensemble**, pas par identité.

Enfin, l'accès à un nœud inexistant lève une `KeyError` nommant le nœud : le moteur ne fabrique pas d'étiquette vide pour un nom inconnu — une absence est une erreur, pas un fait.

### Récapitulatif — la partie ATMS

| Opération | Ce qu'elle fait | Ce qu'elle ne fait pas |
|---|---|---|
| `add_assumption` | pose une hypothèse, étiquette `{nom}` | ne la justifie pas |
| `add_justification` | dérive la conclusion dans le produit cartésien des étiquettes des prémisses, filtré par les nœuds `out` | ne rejoue rien d'existant |
| `invalidate_environment` | retire l'environnement et ses sur-ensembles de toutes les étiquettes | ne redérive rien, ne restaure rien |
| justification vers `⊥` | invalide l'environnement contradictoire | ne mémorise pas le nogood |
| `explain_node` | expose environments + justifications | ne décide rien |

## 6. Expliquer un nœud

Un ATMS sert aussi à **expliquer** : pour un nœud donné, quelles lectures le dérivent et par quelles justifications. `explain_node` renvoie les environnements (en listes triées) et les justifications qui contribuent au nœud — sans dire lesquelles contribuent à *quel* environnement : la structure est celle d'une justification, pas d'un historique.

In [8]:
atms8 = ATMS()
atms8.add_assumption("d")
atms8.add_node("e")
atms8.add_justification(["d"], [], "e")

print("e :", json.dumps(atms8.explain_node("e"), ensure_ascii=False, sort_keys=True))
print("d :", json.dumps(atms8.explain_node("d"), ensure_ascii=False, sort_keys=True))

e : {"environments": [["d"]], "is_assumption": false, "justifications": [{"in_nodes": ["d"], "out_nodes": []}], "name": "e"}
d : {"environments": [["d"]], "is_assumption": true, "justifications": [], "name": "d"}


### Lecture du résultat

`e` n'est pas une hypothèse (`is_assumption` vaut `false`) et sa justification unique référence `d` en entrée, sans nœud `out`. `d` est une hypothèse : `is_assumption` vaut `true` et sa liste de justifications est vide — une hypothèse n'est justifiée par rien, elle est supposée. C'est cette distinction qui permet de lire la chaîne des dépendances d'une conclusion jusqu'aux hypothèses.

## 7. Résoudre un conflit entre agents : les cinq stratégies

Passer d'un nœud à un conflit entre agents : deux inquireurs du même dossier maintiennent des croyances concurrentes, et il faut trancher. Le résolveur expose cinq stratégies, sélectionnables par nom :

| Stratégie | Critère |
|---|---|
| `confidence_based` | la confiance la plus haute |
| `evidence_based` | nombre de justifications × confiance |
| `consensus` | majorité des votes `valid`, à partir de trois agents |
| `agent_expertise` | l'agent dont le rôle correspond au domaine du conflit |
| `temporal` | la croyance la plus récente |

Une stratégie qui ne s'applique pas **replie sur la confiance** en le disant : le résultat porte alors `fallback_from`, le nom de la stratégie demandée.

In [9]:
conflit = {
    "conflict_id": "C-1",
    "context": {"type": "hypothesis"},
    "beliefs": {
        "sherlock": {"belief_name": "H1", "confidence": 0.9, "justification_count": 2},
        "watson": {"belief_name": "H2", "confidence": 0.7, "justification_count": 3},
        "informal": {"belief_name": "H3", "confidence": 0.4, "justification_count": 1},
    },
}

resolveur = ConflictResolver()
for strategie in ConflictResolver.STRATEGIES:
    res = resolveur.resolve(conflit, strategy=strategie)
    print(f"{strategie:17s} resolu={res['resolved']} agent={res['chosen_agent']} "
          f"| {res['reasoning']}")

confidence_based  resolu=True agent=sherlock | Highest confidence: 0.90 by sherlock
evidence_based    resolu=True agent=watson | Best evidence score: 2.10 by watson
consensus         resolu=False agent=None | Consensus tie
agent_expertise   resolu=True agent=sherlock | Expert sherlock for domain hypothesis
temporal          resolu=True agent=sherlock | temporal not applicable (no belief carries a timestamp); fell back to confidence. Highest confidence: 0.90 by sherlock


### Lecture du résultat

Les cinq stratégies ne tranchent pas le même conflit de la même façon : `confidence_based` retient Sherlock (0.90 contre 0.70), `evidence_based` retient Watson (trois justifications contre deux), `consensus` ne tranche pas (aucune croyance ne porte de vote `valid`), `agent_expertise` retient Sherlock (expert du domaine `hypothesis`), et `temporal` retombe sur la confiance en signalant le repli : « temporal not applicable (no belief carries a timestamp) ».

Le message de repli est la partie importante : un repli silencieux donnerait l'illusion qu'une chronologie a été examinée. Ici le résultat affiche la stratégie qui a réellement décidé (`confidence_based`) et le raisonnement qui l'a fait.

## Exercice 2 — quelle stratégie tranche ce conflit ?

**Objectif** : choisir, parmi les cinq stratégies, celle dont le critère donne un choix **différent** de celui de la confiance, sur un conflit neuf.

**Consigne** : le conflit ci-dessous est neuf ; écrivez dans `strategie_attendue` le nom de la stratégie qui y retient `watson`.

`# Indice` : une seule des cinq strategies aboutit à un choix différent de celui de la confiance ; comparez critère par critère plutôt que de calculer les cinq résultats.
`# Etape 1` : eligibility de chaque stratégie (tableau de la section 7).
`# Etape 2` : identifier celle dont le classement diffère de celui des confiances.

In [10]:
# Exercice 2 : quelle strategie tranche ce conflit ?
conflit_ex2 = {
    "conflict_id": "C-EX2",
    "context": {"type": "evidence"},
    "beliefs": {
        "sherlock": {"belief_name": "A", "confidence": 0.8, "justification_count": 1},
        "watson": {"belief_name": "B", "confidence": 0.6, "justification_count": 4},
    },
}

strategie_attendue = None  # TODO etudiant : le nom de la strategie qui retient "watson"

print("Exercice a completer")

Exercice a completer


## 8. Les garde-fous : une stratégie inconnue, un conflit sans croyances, une confiance absente

Trois erreurs de contrat sont refusées bruyamment ou signalées :

- une **stratégie inconnue** lève une `ValueError` qui rappelle la liste des stratégies connues — une faute de frappe ne devient jamais un repli silencieux ;
- un **conflit sans dictionnaire `beliefs`** lève une `ValueError` : une clé mal nommée ne résout personne en silence ;
- une **confiance absente** n'est pas comptée comme zéro. Le moteur refuse de comparer et **nomme** l'agent concerné dans `missing_confidence`.

In [11]:
# 1. Une strategie inconnue est refusee, avec la liste des strategies valides.
try:
    resolveur.resolve(conflit, strategy="magie")
except ValueError as e:
    print("strategie inconnue -> ValueError :", e)

# 2. Un conflit sans dictionnaire "beliefs" est refuse.
try:
    resolveur.resolve({"conflict_id": "C-2"}, strategy="confidence_based")
except ValueError as e:
    print("conflit sans beliefs -> ValueError :", e)

# 3. Une confiance absente n'est pas un zero : le conflit n'est pas resolu.
sans_confiance = {
    "conflict_id": "C-3",
    "beliefs": {
        "sherlock": {"belief_name": "H1", "confidence": 0.9},
        "watson": {"belief_name": "H2"},
    },
}
res = resolveur.resolve(sans_confiance, strategy="confidence_based")
print("conflit sans confiance : resolu =", res["resolved"], "|", res["reasoning"])
print("agents sans confiance :", res["missing_confidence"])

strategie inconnue -> ValueError : Unknown conflict resolution strategy 'magie'; known: confidence_based, evidence_based, consensus, agent_expertise, temporal
conflit sans beliefs -> ValueError : Conflict has no 'beliefs' mapping (agent -> belief dict); got keys ['conflict_id']
conflit sans confiance : resolu = False | No confidence for watson: the beliefs cannot be compared
agents sans confiance : ['watson']


### Lecture du résultat

Les deux premières lignes sont des `ValueError` : le message nomme la stratégie demandée et rappelle les cinq stratégies valides ; le second nomme la clé manquante et montre les clés effectivement présentes. Aucun repli, aucune valeur par défaut inventée.

La troisième ligne est le garde-fou le plus important : Sherlock a 0.90 et Watson n'a **pas** de confiance. Le moteur ne traite pas l'absence comme un 0.0 (ce qui donnerait la victoire à Sherlock par défaut) : il déclare le conflit non résolu et nomme l'agent fautif dans `missing_confidence`. Le raisonnement affiché — « No confidence for watson: the beliefs cannot be compared » — dit pourquoi, pas seulement quoi.

## Exercice 3 — un conflit dont une croyance n'a pas de confiance

**Objectif** : prédire la réponse du moteur sur un conflit où une croyance est chiffrée et l'autre non.

**Consigne** : écrivez dans `repense_attendue` ce que le moteur renvoie (résolu ou non, et le nom de l'agent sans confiance), puis comparez avec la mesure.

`# Indice` : le moteur ne compare pas une confiance absente à zéro ; il nomme l'agent concerné et déclare le conflit non résolu.
`# Etape 1` : compter les agents dont la croyance porte une valeur numérique de `confidence`.
`# Etape 2` : écrire la réponse attendue (le booléen `resolved` et la liste `missing_confidence`).

In [12]:
# Exercice 3 : un conflit dont une croyance n'a pas de confiance
conflit_ex3 = {
    "conflict_id": "C-EX3",
    "beliefs": {
        "sherlock": {"belief_name": "X", "confidence": 0.5},
        "watson": {"belief_name": "Y"},
    },
}

repense_attendue = None  # TODO etudiant : (resolved, missing_confidence)

print("Exercice a completer")

Exercice a completer


## 9. Consensus, expertise, chronologie : trois limites lues

Les trois stratégies restantes ont chacune une condition d'application, et chacune de ces conditions se lit dans le résultat :

- `consensus` exige **au moins trois agents** : avec deux, il refuse de trancher ;
- `agent_expertise` exige un **domaine mappé** : un domaine inconnu n'a aucun expert, et le repli est nommé par `fallback_from` ;
- `temporal` exige qu'une croyance porte un **horodatage** : sinon il retombe sur la confiance.

In [13]:
# Consensus : la condition est le nombre d'agents.
deux_agents = {
    "conflict_id": "C-4",
    "beliefs": {
        "a": {"belief_name": "X", "confidence": 0.9, "valid": True},
        "b": {"belief_name": "Y", "confidence": 0.5, "valid": False},
    },
}
res = resolveur.resolve(deux_agents, strategy="consensus")
print("consensus a 2 agents :", res["resolved"], "|", res["reasoning"])

# Expertise : le domaine est mappe vers un role d'agent.
res = resolveur.resolve(conflit, strategy="agent_expertise")
print("expertise (domaine connu) :", res["chosen_agent"], "|", res["reasoning"])

res = resolveur.resolve({**conflit, "context": {"type": "inconnu"}},
                        strategy="agent_expertise")
print("expertise (domaine inconnu) :", res["chosen_agent"], "|", res["reasoning"])
print("repli depuis :", res["fallback_from"])

# Temporal : la croyance la plus recente l'emporte.
avec_temps = {
    "conflict_id": "C-5",
    "beliefs": {
        "sherlock": {"belief_name": "H1", "confidence": 0.3,
                     "timestamp": "2026-10-01T10:00:00"},
        "watson": {"belief_name": "H2", "confidence": 0.9,
                   "timestamp": "2026-10-28T09:00:00"},
    },
}
res = resolveur.resolve(avec_temps, strategy="temporal")
print("temporal :", res["chosen_agent"], "|", res["reasoning"])

consensus a 2 agents : False | Consensus requires 3+ agents
expertise (domaine connu) : sherlock | Expert sherlock for domain hypothesis
expertise (domaine inconnu) : sherlock | agent_expertise not applicable (no expert mapped for 'inconnu'); fell back to confidence. Highest confidence: 0.90 by sherlock
repli depuis : agent_expertise
temporal : watson | Most recent: watson at 2026-10-28T09:00:00


### Lecture du résultat

Le consensus refuse de trancher avec deux agents — même quand les deux votes sont opposés et nets — et le dit : « Consensus requires 3+ agents ». Le seuil n'est pas un détail d'implémentation, c'est la condition de validité du vote majoritaire.

L'expertise fonctionne quand le domaine est mappé (le domaine `hypothesis` désigne Sherlock comme expert). Sur un domaine inconnu, le moteur ne désigne personne : il le dit (« no expert mapped for 'inconnu' »), replie sur la confiance, et **nomme la stratégie qui a été demandée** dans `fallback_from`.

La chronologie, enfin, choisit la croyance la plus récente (Watson, datée du 28 octobre) alors que sa confiance est la plus faible : c'est bien l'horodatage qui décide, pas la confiance.

## 10. L'historique du résolveur

Le résolveur conserve chaque résolution : l'historique alimente des statistiques par stratégie. C'est la trace qui permet de répondre « sur quoi avons-nous vraiment tranché, et avec quelle règle ».

In [14]:
stats = resolveur.get_stats()
print("conflits traites :", stats["total_conflicts"])
print("resolus :", stats["resolved"])
print("non resolus :", stats["unresolved"])
print("par strategie :", stats["by_strategy"])
print("longueur de l'historique :", len(resolveur.get_history()))

conflits traites : 10
resolus : 7
non resolus : 3
par strategie : {'confidence_based': 4, 'evidence_based': 1, 'consensus': 2, 'agent_expertise': 2, 'temporal': 1}
longueur de l'historique : 10


### Lecture du résultat

Les compteurs reflètent les résolutions de ce carnet, y compris les refus et les replis : un conflit non résolu (le consensus sans vote, le consensus à deux agents, le conflit sans confiance) compte dans `total_conflicts` et dans `unresolved`. Le décompte par stratégie montre que les replis sont comptés sous la stratégie **réellement employée** (`confidence_based`), pas sous celle qui était demandée : le repli est une trace, pas une stratégie fantôme.

Les deux appels refusés par `ValueError` ne sont **pas** comptés : l'exception est levée avant l'enregistrement, et l'historique ne retient que les conflits réellement passés au résolveur. Le compte « non résolus » ne contient donc que des conflits **traités mais non tranchés**, jamais un conflit refusé.

### Récapitulatif — la partie conflits

| Stratégie | Condition d'application | Ce qu'elle mesure |
|---|---|---|
| `confidence_based` | une confiance numérique par croyance | la confiance la plus haute |
| `evidence_based` | idem | justifications × confiance |
| `consensus` | au moins trois agents | majorité des votes `valid` |
| `agent_expertise` | un domaine mappé vers un rôle | l'agent expert du domaine |
| `temporal` | un horodatage par croyance | la croyance la plus récente |
| repli | stratégie inapplicable | la confiance, avec `fallback_from` nommé |


## 11. Ce que ce carnet a établi

| Point | Moteur | Ce qui est mesuré |
|---|---|---|
| Une étiquette ATMS est un ensemble d'environnements, pas un booléen | `_atms_core` | `z` porte deux environnements, `w` en hérite |
| Une conclusion combine ses prémisses par produit cartésien puis union | `_atms_core` | `w` = `['p', 'q']` et `['p', 'q', 's']` |
| Un nœud `out` bloque par sous-ensemble | `_atms_core` | cas 1 dérivé, cas 2 écarté |
| L'invalidation retranche, elle ne rederive pas | `_atms_core` | `t` vidé, justification toujours présente |
| Une contradiction n'est pas mémorisée | `_atms_core` | `{m, n}` cohérent après le nogood |
| Une stratégie inapplicable se replie en le disant | `_conflict_resolution` | `fallback_from` nommé |
| Une confiance absente n'est pas un zéro | `_conflict_resolution` | `resolved` faux, `missing_confidence` nommé |
| Une faute de contrat échoue bruyamment | `_conflict_resolution` | `ValueError` sur stratégie inconnue et sur `beliefs` absent |

Les deux moteurs sont **inchangés** : ce carnet les mesure, il ne les réécrit pas. Pour la version booléenne du même problème, voir le rung [6-JTMS](./Argumentation-06-JTMS-Python.ipynb) ; pour la composition d'agents qui consume ces conflits, le rung [7-Orchestration](./Argumentation-07-Orchestration-Python.ipynb).

**À retenir**

- Une croyance ATMS n'est pas vraie ou fausse : elle est dérivable **sous un ensemble d'environnements**, et cet ensemble se lit, se compare et s'invalide.
- L'invalidation est une perte, pas un retour en arrière : le moteur ne rejoue aucune justification, et une contradiction qu'il a produite ne reste pas en mémoire.
- Un conflit entre agents se tranche par une stratégie **nommée**, et toute stratégie inapplicable **le dit** plutôt que de trancher silencieusement.

**Vérifiez votre compréhension**

1. Pourquoi l'étiquette de `n` survit-elle à l'invalidation de `{m, n}` alors que celle de `m` est vidée ? *(réponse : l'invalidation retire l'environnement et ses sur-ensembles ; `{n}` n'est pas un sur-ensemble de `{m, n}`)*
2. Que se passe-t-il si l'on invalide `{p}` dans un ATMS où `p` porte les environnements `{p}` et `{p, s}` ? *(réponse : les deux sont retirés, car `{p, s}` contient `{p}`)*
3. Pourquoi la stratégie `temporal` ne peut-elle pas être utilisée sur le conflit de la section 7 ? *(réponse : aucune croyance ne porte d'horodatage ; le repli sur la confiance est nommé dans le raisonnement)*
4. Que changerait le fait de compter une confiance absente comme zéro ? *(réponse : l'agent le moins confiant perdrait par défaut ; le moteur refuse cette comparaison et nomme l'agent concerné)*

**Pour aller plus loin**

- Le rung [6-JTMS](./Argumentation-06-JTMS-Python.ipynb) pour l'étiquetage booléen et la cascade de rétractation, et le carnet de révision de croyances pour l'axiomatisation AGM.
- Le rung [7-Orchestration](./Argumentation-07-Orchestration-Python.ipynb) pour la composition des agents dont ce résolveur tranche les conflits.
- Le dépôt EPITA (`2025-Epita-Intelligence-Symbolique`, `argumentation_analysis/services/jtms/`) pour le moteur amont et ses tests.